# 📊 Polynomial Regression Explanation and Hands-on Example

Welcome to the hands-on explanation notebook for **Polynomial Regression**! In this notebook, we will:
1. Generate a synthetic curved dataset (non-linear relationship) with noise.
2. Fit a standard **Linear Regression** model to show why a straight line underfits the data.
3. Apply polynomial feature transformation to fit a curve.
4. Compare models of different polynomial degrees (underfitting, ideal fitting, and overfitting).
5. Implement polynomial feature expansion and solve it from scratch using the **Normal Equation**:
   $$\mathbf{w} = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$$

Let's begin by importing the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Set seed for reproducibility
np.random.seed(42)

## 1. Data Generation

We will generate synthetic data using a quadratic equation:
$$y = 0.5x^2 - 2.0x + 3.0 + \epsilon$$

Where:
*   True quadratic weight ($w_2$) = 0.5
*   True linear weight ($w_1$) = -2.0
*   True bias ($w_0$) = 3.0
*   $\epsilon$ is random Gaussian noise.

In [ ]:
# Generate 80 random points between -2 and 4
X = np.random.rand(80, 1) * 6 - 2
y = 0.5 * (X**2) - 2.0 * X + 3.0 + np.random.randn(80, 1) * 0.5

# Sort X and y for smoother plotting later
sort_idx = np.argsort(X.flatten())
X_sorted = X[sort_idx]
y_sorted = y[sort_idx]

# Plot the generated data
plt.figure(figsize=(8, 5))
plt.scatter(X, y, color='blue', alpha=0.6, label='Data Points')
plt.xlabel('X (Independent Variable)')
plt.ylabel('y (Dependent Variable)')
plt.title('Non-Linear Synthetic Dataset')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.show()

## 2. Linear vs. Polynomial Fits (Bias-Variance Tradeoff)

If we fit a straight line (Degree 1), the model cannot capture the curve (high bias, underfitting).
If we fit a very high degree polynomial (e.g., Degree 15), the model will capture the noise (high variance, overfitting).
Let's see this visually by fitting polynomials of degree 1, 2, and 15.

In [ ]:
degrees = [1, 2, 15]
plt.figure(figsize=(12, 6))
plt.scatter(X, y, color='blue', alpha=0.5, label='Data Points')

for degree in degrees:
    # 1. Transform X to polynomial features
    poly_features = PolynomialFeatures(degree=degree, include_bias=False)
    X_poly = poly_features.fit_transform(X)
    
    # 2. Fit standard Linear Regression on transformed features
    model = LinearRegression()
    model.fit(X_poly, y)
    
    # 3. Predict on sorted X for line plotting
    X_poly_sorted = poly_features.transform(X_sorted)
    y_pred = model.predict(X_poly_sorted)
    
    # Calculate R2 score
    r2 = r2_score(y, model.predict(X_poly))
    
    plt.plot(X_sorted, y_pred, label=f'Degree {degree} (R² = {r2:.3f})', linewidth=2)

plt.xlabel('X')
plt.ylabel('y')
plt.title('Underfitting vs. Ideal Fit vs. Overfitting')
plt.ylim(-1, 12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 3. Polynomial Regression from Scratch (Normal Equation)

For multiple linear regression or polynomial regression, we can find the optimal parameters analytically using the **Normal Equation**:
$$\mathbf{w} = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$$

Where:
*   $\mathbf{X}$ is the feature matrix, where each row is $[1, x, x^2, \dots, x^n]$.
*   $\mathbf{y}$ is the target vector.
*   $\mathbf{w}$ is the weight vector $[w_0, w_1, w_2, \dots, w_n]^T$.

Let's implement this feature expansion and optimization in pure NumPy.

In [ ]:
def get_polynomial_features(X, degree):
    """
    Manually create polynomial features including bias (column of 1s).
    """
    m = len(X)
    # Column of ones for bias intercept
    X_poly = np.ones((m, 1))
    
    # Add powers of X
    for power in range(1, degree + 1):
        X_poly = np.hstack((X_poly, X ** power))
        
    return X_poly

def solve_normal_equation(X, y):
    """
    Solve for w using normal equation: w = (X^T * X)^-1 * X^T * y
    """
    XTX = X.T @ X
    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv @ X.T @ y
    return w

# 1. Transform features to degree 2 (quadratic)
X_poly_scratch = get_polynomial_features(X, degree=2)

# 2. Solve using Normal Equation
weights_scratch = solve_normal_equation(X_poly_scratch, y)

print("Learned Polynomial Coefficients (Scratch):")
for i, w in enumerate(weights_scratch.flatten()):
    print(f"w_{i} (coefficient for X^{i}): {w:.4f}")

# Compare with true coefficients (0.5x^2 - 2x + 3)
print(f"\nTrue Coefficients: w_0 = 3.0, w_1 = -2.0, w_2 = 0.5")

## 4. Model Evaluation

Let's predict using our scratch weights and check the Mean Squared Error (MSE) and R-squared ($R^2$) score.

In [ ]:
# Predict using scratch weights
X_poly_sorted_scratch = get_polynomial_features(X_sorted, degree=2)
y_pred_scratch = X_poly_sorted_scratch @ weights_scratch

# Calculate overall metrics
y_pred_all_scratch = X_poly_scratch @ weights_scratch
mse_scratch = mean_squared_error(y, y_pred_all_scratch)
r2_scratch = r2_score(y, y_pred_all_scratch)

print(f"Scratch Model (Degree 2) MSE: {mse_scratch:.4f}")
print(f"Scratch Model (Degree 2) R2 Score: {r2_scratch:.4f}")

# Plot scratch fit
plt.figure(figsize=(8, 5))
plt.scatter(X, y, color='blue', alpha=0.6, label='Data Points')
plt.plot(X_sorted, y_pred_scratch, color='red', linewidth=2.5, label='Custom Polynomial Fit')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Custom Polynomial Regression Fit (Normal Equation)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.show()

## 💡 Connection to Computer Vision & YOLO
*   **Object Tracking Trajectories:** Trackers like DeepSORT or ByteTrack model bounding box positions over time. Bounding box coordinates $x(t)$ and $y(t)$ usually follow a curve rather than a line. Modeling these trajectories using second or third-degree polynomial regression allows trackers to predict future bounding box location with higher accuracy during occlusions.